In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, ArrayType

In [2]:
spark = SparkSession.builder.appName("NYC-traffic-collision").getOrCreate()

# Reading raw data

### Nyc traffic

In [3]:
df_nyc_traffic = spark.read\
.option("delimiter",",")\
.option("header", True)\
.option("quote", '"')\
.option("escape", '"')\
.option("multiline", True)\
.option("header", True)\
.csv("dataset/Motor_Vehicle_Collisions_-_Crashes_2024.csv")
df_nyc_traffic.show(2, truncate=False)

+----------+----------+-------+--------+--------+---------+---------------------+-----------------------+-----------------+---------------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+------------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+------------+-------------------+-------------------+-------------------+-------------------+-------------------+----------+
|CRASH DATE|CRASH TIME|BOROUGH|ZIP CODE|LATITUDE|LONGITUDE|LOCATION             |ON STREET NAME         |CROSS STREET NAME|OFF STREET NAME|NUMBER OF PERSONS INJURED|NUMBER OF PERSONS KILLED|NUMBER OF PEDESTRIANS INJURED|NUMBER OF PEDESTRIANS KILLED|NUMBER OF CYCLIST INJURED|NUMBER OF CYCLIST KILLED|NUMBER OF MOTORIST INJURED|NUMBER OF MOTORIST KILLED|CONTRIBUTING FACTO

In [4]:
df_nyc_traffic.count()

91314

### Holiday days

In [5]:
import requests
import json

def fetch_data(url: str):
    response = requests.get(url)
    response.raise_for_status()
    return  response.json()


year = 2024
country_code = 'US'
pub_holidays = fetch_data(f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_code}")

In [6]:
schema = StructType([
    StructField("date", StringType(), True),
    StructField("localName", StringType(), True),
    StructField("name", StringType(), True),
    StructField("countryCode", StringType(), True),
    StructField("fixed", BooleanType(), True),
    StructField("global", BooleanType(), True),
    StructField("counties", ArrayType(StringType()), True),
    StructField("launchYear", IntegerType(), True),
    StructField("types", ArrayType(StringType()), True),
])

In [7]:
df_holidays = spark.createDataFrame(pub_holidays,schema)

In [8]:
df_holidays.show(truncate=False)

+----------+------------------------------------+------------------------------------+-----------+-----+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------------------+
|date      |localName                           |name                                |countryCode|fixed|global|counties                                                                                                                                                                                                                               |launchYear|types                |
+----------+------------------------------------+------------------------------------+-----------+-----+------+---------------------------------------------------------------------------------------------------------------------------------------

# Cleaning data and adapt schema

### nyc

In [9]:
relevant_cols = ["CRASH_DATE","BOROUGH","ZIP CODE","NUMBER_OF_PERSONS_INJURED","NUMBER_OF_PERSONS_KILLED","NUMBER_OF_PEDESTRIANS_INJURED","NUMBER_OF_PEDESTRIANS_KILLED",
                 "NUMBER_OF_CYCLIST_INJURED","NUMBER_OF_CYCLIST_KILLED","NUMBER_OF_MOTORIST_INJURED","NUMBER_OF_MOTORIST_KILLED","TIME_OF_DAY"]
'''
part_of_day
early-morning: 8 - 10
late-morning: 11 - 13 
early-afternoon: 14 - 16
late-afternoon: 17 - 19
evening: 20 - 22
night 22 - 7
'''
df_nyc_traffic_l1 = df_nyc_traffic\
.withColumn("CRASH_DATE",f.to_date(f.col("CRASH DATE"),"MM/dd/yyyy"))\
.filter(f.col("CRASH_DATE").isNotNull())\
.filter( (f.col("CRASH_DATE") >= '2024-01-01') & (f.col("CRASH_DATE") <= '2024-12-31'))\
.withColumn("NUMBER_OF_PERSONS_INJURED",f.col("NUMBER OF PERSONS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PERSONS_KILLED",f.col("NUMBER OF PERSONS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_INJURED",f.col("NUMBER OF PEDESTRIANS INJURED").cast("integer"))\
.withColumn("NUMBER_OF_PEDESTRIANS_KILLED",f.col("NUMBER OF PEDESTRIANS KILLED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_INJURED",f.col("NUMBER OF CYCLIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_CYCLIST_KILLED",f.col("NUMBER OF CYCLIST KILLED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_INJURED",f.col("NUMBER OF MOTORIST INJURED").cast("integer"))\
.withColumn("NUMBER_OF_MOTORIST_KILLED",f.col("NUMBER OF MOTORIST KILLED").cast("integer"))\
.fillna(0)\
.withColumn("CRASH_HOUR", f.hour(f.to_timestamp(f.col("CRASH TIME"),"H:m")))\
.withColumn("TIME_OF_DAY", f.when( (f.col("CRASH_HOUR") >= 8) & (f.col("CRASH_HOUR") < 11),"early-morning")
                        .when( (f.col("CRASH_HOUR") >= 11) & (f.col("CRASH_HOUR") < 14),"late-morning")\
                        .when( (f.col("CRASH_HOUR") >= 14) & (f.col("CRASH_HOUR") < 17),"early-afternoon")\
                        .when( (f.col("CRASH_HOUR") >= 17) & (f.col("CRASH_HOUR") < 20),"late-afternoon")\
                        .when( (f.col("CRASH_HOUR") >= 20) & (f.col("CRASH_HOUR") < 23),"evening")\
                        .when( (f.col("CRASH_HOUR") >= 23) & (f.col("CRASH_HOUR") < 8),"night")\
                        .otherwise(None)\
          )\
.select(relevant_cols)

In [10]:
df_nyc_traffic_l1.count()

91314

In [11]:
df_nyc_traffic_l1.show()

+----------+-------+--------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+---------------+
|CRASH_DATE|BOROUGH|ZIP CODE|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|    TIME_OF_DAY|
+----------+-------+--------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+---------------+
|2024-09-13|   NULL|    NULL|                        0|                       0|                            0|                           0|                        0|                       0|                         0|   

### holidays

In [12]:
relevant_holidays_cols = ["date","isPublic","isLocal"]

df_holidays_l1 = df_holidays\
.withColumn("isPublic", f.array_contains(f.col("types"),"Public"))\
.withColumn("isLocal", (f.col("counties").isNotNull() & f.array_contains(f.col("counties"),"US-NY")))\
.select(relevant_holidays_cols)\

df_holidays_l1.show(truncate=False)

+----------+--------+-------+
|date      |isPublic|isLocal|
+----------+--------+-------+
|2024-01-01|true    |false  |
|2024-01-15|true    |false  |
|2024-02-12|false   |true   |
|2024-02-19|true    |false  |
|2024-03-29|true    |false  |
|2024-03-29|false   |false  |
|2024-05-08|false   |false  |
|2024-05-27|true    |false  |
|2024-06-19|true    |false  |
|2024-07-04|true    |false  |
|2024-09-02|true    |false  |
|2024-10-14|true    |true   |
|2024-10-14|true    |false  |
|2024-11-11|true    |false  |
|2024-11-28|true    |false  |
|2024-12-25|true    |false  |
+----------+--------+-------+



### Join data

In [13]:
df_nyc_traffic_l2 = df_nyc_traffic_l1.join(f.broadcast(df_holidays_l1), df_nyc_traffic_l1.CRASH_DATE==df_holidays_l1.date, how="left")\
.withColumn("isHoliday",f.col("date").isNotNull())\
.drop("date")\
.groupBy("CRASH_DATE","BOROUGH","ZIP CODE","TIME_OF_DAY","isPublic","isLocal","isHoliday")\
.agg(f.sum("NUMBER_OF_PERSONS_INJURED").alias("NUMBER_OF_PERSONS_INJURED"),
     f.sum("NUMBER_OF_PERSONS_KILLED").alias("NUMBER_OF_PERSONS_KILLED"),
     f.sum("NUMBER_OF_PEDESTRIANS_INJURED").alias("NUMBER_OF_PEDESTRIANS_INJURED"),
     f.sum("NUMBER_OF_PEDESTRIANS_KILLED").alias("NUMBER_OF_PEDESTRIANS_KILLED"),
     f.sum("NUMBER_OF_CYCLIST_INJURED").alias("NUMBER_OF_CYCLIST_INJURED"),
     f.sum("NUMBER_OF_CYCLIST_KILLED").alias("NUMBER_OF_CYCLIST_KILLED"),
     f.sum("NUMBER_OF_MOTORIST_INJURED").alias("NUMBER_OF_MOTORIST_INJURED"),
     f.sum("NUMBER_OF_MOTORIST_KILLED").alias("NUMBER_OF_MOTORIST_KILLED")
    )

## Persisting data

In [14]:
df_nyc_traffic_l2\
.write\
.mode("overwrite")\
.option("compression", "snappy") \
.parquet("output")

In [15]:
df_nyc_traffic_l2.filter("isHoliday == 'true'").count()

2216

In [16]:
df_nyc_traffic_l2.show()

+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|CRASH_DATE|      BOROUGH|ZIP CODE|    TIME_OF_DAY|isPublic|isLocal|isHoliday|NUMBER_OF_PERSONS_INJURED|NUMBER_OF_PERSONS_KILLED|NUMBER_OF_PEDESTRIANS_INJURED|NUMBER_OF_PEDESTRIANS_KILLED|NUMBER_OF_CYCLIST_INJURED|NUMBER_OF_CYCLIST_KILLED|NUMBER_OF_MOTORIST_INJURED|NUMBER_OF_MOTORIST_KILLED|
+----------+-------------+--------+---------------+--------+-------+---------+-------------------------+------------------------+-----------------------------+----------------------------+-------------------------+------------------------+--------------------------+-------------------------+
|2024-05-08|        BRONX|   10460|        evening|   false|  false|     true|                        3|                 

# Some interesting queries for the use case

Analyze if during holidays or weekdays there are more/less/same number of people injured/killed on average per part of the day


In [17]:
df_nyc_traffic_l2\
.groupBy("TIME_OF_DAY","isHoliday").agg(f.round(f.avg("NUMBER_OF_PERSONS_INJURED"),2).alias("AVG_PERSON_INJURED"), f.round(f.avg("NUMBER_OF_PERSONS_KILLED"),4).alias("AVG_PERSONS_KILLED"))\
.orderBy(f.col("isHoliday"),f.col("AVG_PERSON_INJURED").desc(),f.col("AVG_PERSONS_KILLED").desc())\
.select("isHoliday","TIME_OF_DAY","AVG_PERSON_INJURED","AVG_PERSONS_KILLED")\
.show()

+---------+---------------+------------------+------------------+
|isHoliday|    TIME_OF_DAY|AVG_PERSON_INJURED|AVG_PERSONS_KILLED|
+---------+---------------+------------------+------------------+
|    false|        evening|              0.99|            0.0061|
|    false|           NULL|              0.97|            0.0078|
|    false| late-afternoon|              0.96|            0.0036|
|    false|early-afternoon|              0.91|            0.0025|
|    false|   late-morning|              0.82|            0.0027|
|    false|  early-morning|              0.78|            0.0026|
|     true|        evening|              1.11|            0.0233|
|     true|           NULL|              1.04|            0.0182|
|     true| late-afternoon|              0.97|            0.0028|
|     true|early-afternoon|              0.89|            0.0048|
|     true|   late-morning|              0.85|            0.0032|
|     true|  early-morning|               0.8|               0.0|
+---------

During the holidays analyze which are the most dangerous ( ranking ) "zone" for each borough

In [18]:
from pyspark.sql.window import Window

w = Window.partitionBy("BOROUGH").orderBy(f.col("AVG_PERSON_INJURED").desc(),f.col("AVG_PERSONS_KILLED").desc())

df_nyc_traffic_l2\
.filter((f.col("isHoliday")==True) & (f.col("BOROUGH").isNotNull()))\
.groupBy("BOROUGH","ZIP CODE")\
.agg(f.round(f.avg("NUMBER_OF_PERSONS_INJURED"),2).alias("AVG_PERSON_INJURED"), f.round(f.avg("NUMBER_OF_PERSONS_KILLED"),4).alias("AVG_PERSONS_KILLED"))\
.withColumn("rank", f.dense_rank().over(w))\
.orderBy(f.col("BOROUGH"),f.col("rank").asc())\
.select("BOROUGH","ZIP CODE","rank","AVG_PERSON_INJURED","AVG_PERSONS_KILLED")\
.show(100)

+---------+--------+----+------------------+------------------+
|  BOROUGH|ZIP CODE|rank|AVG_PERSON_INJURED|AVG_PERSONS_KILLED|
+---------+--------+----+------------------+------------------+
|    BRONX|   10464|   1|               3.0|               0.0|
|    BRONX|   10455|   2|              1.88|               0.0|
|    BRONX|   10471|   3|               1.4|               0.0|
|    BRONX|   10469|   4|              1.29|               0.0|
|    BRONX|   10459|   5|              1.28|               0.0|
|    BRONX|   10456|   6|              1.23|               0.0|
|    BRONX|   10453|   7|              1.15|               0.0|
|    BRONX|   10475|   8|              1.14|               0.0|
|    BRONX|   10474|   9|               1.0|               0.0|
|    BRONX|   10460|  10|              0.95|               0.0|
|    BRONX|   10458|  11|              0.92|               0.0|
|    BRONX|   10451|  12|              0.86|               0.0|
|    BRONX|   10473|  13|              0